# CHATBOT CIMA | Actividad 8 — Registro de observabilidad en MLflow

**Objetivo:** importar a MLflow las evidencias **SIMULADAS OFFLINE** de GitHub Actions y obtener capturas de experimentos, métricas, parámetros y artefactos. **No monitorea Render en tiempo real ni modifica el modelo.**

**Requisitos:** runtime CPU de Google Colab (no se requiere GPU), conexión para instalar dependencias y clonar GitHub, y el artefacto `cima-a8-simulacion.zip` descargado desde la **última ejecución verde** de GitHub Actions. El entorno de Colab es temporal; exporta el almacén al finalizar.

**Evidencias que debes capturar:** pantalla de runs MLflow, métricas del incidente de latencia, métricas de calidad ficticia y lista de artefactos. No subir información personal o bancaria.

## Celda 1 — Instalación de MLflow

Se instala MLflow sobre Python del entorno en lugar de fijar versiones de scikit-learn del proyecto. **No instalar ni ejecutar los modelos de Fase II aquí.**

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlflow>=3.5,<4'], check=True)
import mlflow
print('Python:',sys.version.split()[0], '| MLflow:',mlflow.__version__)

## Celda 2 — Obtener el código publicado en GitHub

Verifica antes que `Actividad8/src/mlflow_registro.py` ya esté publicado. Si no existe, súbelo desde el paquete de actualización.

In [ ]:
from pathlib import Path
repo=Path('/content/deep-learning-experiments')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/alfredosalazar-ai/deep-learning-experiments.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
project=repo/'Actividad8'
script=project/'src'/'mlflow_registro.py'
assert script.is_file(),f'Archivo faltante en GitHub: {script}. Publícalo antes de continuar.'
print('Código disponible:',script)

## Celda 3 — Cargar el ZIP de la ejecución exitosa

En GitHub → Actions → última ejecución **Success** → Artifacts → descarga `cima-a8-simulacion`. **Sube el archivo ZIP, sin descomprimirlo.** Es la fuente de verdad de esta demostración.

In [ ]:
from google.colab import files
uploaded=files.upload()
assert len(uploaded)==1,'Sube únicamente el ZIP de evidencia.'
name=next(iter(uploaded))
assert name.lower().endswith('.zip'),'Se requiere un archivo ZIP.'
zip_path=Path('/content')/name
if not zip_path.exists():
    zip_path.write_bytes(uploaded[name])
print('Artefacto recibido:',zip_path.name, '| tamaño:',zip_path.stat().st_size,'bytes')

## Celda 4 — Registrar cinco runs en MLflow

Se registran 3 incidentes (uno por run), 1 simulación de calidad ficticia y 1 run de alertas/runbooks. `--workflow-run` y `--commit` son **referencias declaradas por el usuario**, no verificación automática del enlace de GitHub.

In [ ]:
tracking_dir=Path('/content/cima_mlflow')
cmd=[sys.executable,'-m','src.mlflow_registro','--input',str(zip_path),'--tracking-dir',str(tracking_dir),'--workflow-run','4','--commit','c0d6991']
subprocess.run(cmd,cwd=str(project),check=True)
print('Registro concluido. Los resultados son SIMULACIONES OFFLINE.')

## Celda 5 — Verificación de runs y métricas (captura 1)

La tabla se obtiene **del almacén MLflow registrado en esta sesión**, no de valores insertados manualmente.

In [ ]:
from IPython.display import display
mlflow.set_tracking_uri((tracking_dir/'mlruns').as_uri())
runs=mlflow.search_runs(experiment_names=['CIMA_Actividad8_Observabilidad_Simulada'])
assert len(runs)>=5,f'Se esperaban al menos 5 runs; hay {len(runs)}'
campos=[c for c in ['tags.mlflow.runName','tags.tipo_run','metrics.latencia_p95_ms','metrics.drift_psi','metrics.error_5xx_rate','metrics.accuracy','metrics.f1_macro','metrics.runbooks_recuperacion_verificada'] if c in runs.columns]
display(runs[campos].fillna('—'))
print('Runs de esta sesión:',len(runs), '| Fuente: SIMULACION_OFFLINE')

## Celda 6 — Mostrar la interfaz MLflow (capturas 2 a 4)

Abre la ventana de MLflow y captura la pestaña **Runs**, las métricas de `INC-01_incidente_offline`, el run de calidad y los artefactos adjuntos.

En Colab puede abrirse una ventana mediante el proxy del kernel. Si la interfaz no carga, utiliza la tabla de la celda anterior como evidencia y conserva la salida del script. No expongas MLflow en un servidor público.

In [ ]:
import subprocess, time
from google.colab import output
port=5000
server=subprocess.Popen(['mlflow','ui','--backend-store-uri',(tracking_dir/'mlruns').as_uri(),'--host','127.0.0.1','--port',str(port)],stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
time.sleep(5)
print('Proceso MLflow:',server.pid,'| activo:',server.poll() is None)
if server.poll() is not None:
    print('MLflow UI no inició; usar tabla de verificación de la celda anterior.')
else:
    output.serve_kernel_port_as_window(port)

## Celda 7 — Respaldo antes de cerrar Colab

MLflow almacena los runs en el entorno temporal; descarga el respaldo para no perder el registro. **No subir el almacén completo a un repositorio público sin revisar su contenido.** En GitHub basta publicar capturas revisadas y el resumen JSON.

In [ ]:
import shutil
resumen=tracking_dir/'resumen_registro_mlflow.json'
assert resumen.is_file(),'Falta el resumen de registro.'
backup=shutil.make_archive('/content/CIMA_A8_MLflow_Tracking_SIMULADO','zip',root_dir=str(tracking_dir))
print('Respaldo generado:',backup)
files.download(str(resumen))
files.download(backup)

## Interpretación académica

Los resultados **son datos artificiales** reutilizados de GitHub Actions. MLflow demuestra que tenemos trazabilidad de los experimentos: parámetros, métricas, runs y artefactos asociados. No demuestra que exista observabilidad continua ni accuracy real de CIMA.

**Checklist de evidencia:** 5 runs visibles; INC-01 presenta latencia antes/durante/después; calidad ficticia indica caída de accuracy/F1; alertas y runbooks tienen artefactos; resumen JSON incluye hash SHA-256 y ubicación de origen. Para trazas distribuidas reales haría falta instrumentar el backend FastAPI, trabajo fuera de este laboratorio.

Fuente: [MLflow Tracking](https://mlflow.org/docs/latest/tracking/) · [Proyecto GitHub](https://github.com/alfredosalazar-ai/deep-learning-experiments/tree/main/Actividad8).